# Curated KG vs original KG: same test pairs

Compares NetMedGPT trained on the original splits (existing checkpoints) with NetMedGPT trained on the curated KG
(`train.py --inference *_curated`), on **identical test pairs**: the curated test edges and their negatives, built as in
`other/evaluate_checkpoints.py` on the curated KG. For the original models the pairs are mapped to the full-KG node
indices by node ID; Recall@100 ranks the same candidate drugs / proteins / effects (those of the curated KG) for both.
Metrics per relation: AUROC, AUPRC, Recall@100; paired t-test over seeds.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "curated_kg_training":
    os.chdir("../..")
import sys
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "other"))
import json
import pickle
import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score

from netmedgpt.model import TransformerModel
from netmedgpt.prepare_txgnn_splits import prepare_txgnn_data, normalize_id
from netmedgpt.utilities import sim, node_level_eval
from evaluate_checkpoints import build_test_edges

RUNS = {   # inference -> (curated split folder, original checkpoint name, relations, node types)
    "random_link_split": ("random_{seed}", "netmedgpt_randomLinkSplit_seed{seed}",
                          ["indication", "contraindication", "off-label use", "drug_protein", "drug_effect"],
                          ["disease|drug", "disease|drug", "disease|drug", "drug|gene/protein", "drug|effect/phenotype"]),
    "zero_shot_split": ("complex_disease_{seed}", "netmedgpt_zeroShot_seed{seed}",
                        ["indication", "contraindication"], ["disease|drug", "disease|drug"]),
}
SEEDS = [1, 2, 3, 4, 5]
CKPT_DIR = "data/model_checkpoints/saved_models_evaluation"
GPU = 0
OUT_DIR = "data/new_analysis/curated_kg_training"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── The two KGs (vocabulary, features) ───────────────────────────────────────
all_param = json.load(open("data/parameters.json"))
hp = all_param["best_hyperparam"]
seq_len = all_param["node2vec"]["walk_length"] * 2 - 1
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
relation_node_types = pickle.load(open("data/KG/relation_node_types.pkl", "rb"))


def load_kg(nodes_f, edges_f, feat_f):
    nodes = pd.read_csv(nodes_f, sep=",")
    edge = pd.read_csv(edges_f)
    kg = {"nodes": nodes, "edge": edge, "feat": torch.load(feat_f), "mask_token": int(edge["z_index"].max()) + 1}
    kg["rel_token"] = edge.drop_duplicates("relation").set_index("relation")["z_index"].astype(int).to_dict()
    kg["relation_mask_index"] = pd.concat([edge[["relation", "z_index"]].drop_duplicates(),
                                           pd.DataFrame([["mask", kg["mask_token"]]], columns=["relation", "z_index"])],
                                          ignore_index=True)
    kg["entity"] = list(nodes["node_type"].unique()) + list(edge["relation"].unique()) + ["mask"]
    kg["key"] = list(zip(nodes["node_type"], nodes["node_id"].astype(str).map(normalize_id)))
    kg["index_of"] = dict(zip(kg["key"], nodes["node_index"]))   # (type, id) -> index, as prepare_txgnn_data maps the splits
    return kg


full = load_kg("data/KG/nodes.csv", "data/KG/edges.csv", "data/KG/embeddings.pt")
cur = load_kg("data/KG/nodes_preprocessed.csv", "data/KG/edges_preprocessed.csv", "data/KG/embeddings_preprocessedKG.pt")
# curated node index -> full-KG node index (by node type and ID; a few nodes share type and ID)
cur_to_full = {i: full["index_of"][k] for i, k in zip(cur["nodes"]["node_index"], cur["key"])}


def load_model(path, kg):
    ckpt = torch.load(path, map_location=device)
    if "model_state" in ckpt:
        state, h, sl = ckpt["model_state"], ckpt.get("hyperparams", hp), ckpt.get("seq_len", seq_len)
    elif "model_state_dict" in ckpt:
        state, h, sl = ckpt["model_state_dict"], ckpt.get("parameters", hp), seq_len
    else:
        state, h, sl = ckpt, hp, seq_len
    model = TransformerModel(kg["mask_token"] + 1, h["hidden_channels"], h["nhead"], h["N_encoder_layers"], sl,
                             device=device, feat=kg["feat"], nodes=kg["nodes"], entity=kg["entity"],
                             relation_mask_index=kg["relation_mask_index"], pos_emb="fixed").to(device)
    model.load_state_dict(state)
    return model.eval(), h, sl


def evaluate(model, h, sl, kg, idx, label, flag, relations, candidates):
    """AUROC / AUPRC per relation on the pairs, Recall@100 over the candidate tails of each relation"""
    pred = sim(idx, model, batch_size=h["batch_size"], method="prob", device=device, mask_token=kg["mask_token"], seq_len=sl)
    all_ids_node2 = {kg["rel_token"][r]: torch.tensor(sorted(candidates[r]), dtype=torch.long) for r in relations}
    recall, _ = node_level_eval(idx, label, all_ids_node2, model, device=device, mask_token=kg["mask_token"], seq_len=sl)
    out = {}
    for i, r in enumerate(relations):
        m = (flag == i + 1).numpy()
        out[r] = {"AUROC": roc_auc_score(label[m], pred[m]), "AUPRC": average_precision_score(label[m], pred[m]),
                  "Recall@100": recall[i]}
    return out

In [ ]:
# ── Score both models on the curated test pairs ──────────────────────────────
rows = []
for inference, (folder, old_name, relations, node_types) in RUNS.items():
    for seed in SEEDS:
        new_ckpt = os.path.join(CKPT_DIR, f"netmedgpt_{inference}_curated_seed{seed}.pt")
        if not os.path.exists(new_ckpt):
            print(f"skip {inference} seed {seed}: {new_ckpt} not found (train it first)")
            continue
        split_dir = os.path.join("data/splits_curated", folder.format(seed=seed))
        splits = [pd.read_csv(os.path.join(split_dir, f), low_memory=False) for f in ["train.csv", "valid.csv", "test.csv"]]
        _, _, test = prepare_txgnn_data(cur["nodes"].copy(), *splits, dict(zip(relations, node_types)))

        # curated test pairs (positives + negatives), as in evaluate_checkpoints.py
        idx, label, flag = build_test_edges(cur["edge"], test, relations, node_types, relation_node_types, seed)
        candidates = {r: cur["edge"].loc[(cur["edge"]["relation"] == r) & (cur["edge"]["node_types"] == nt), "y_index"].unique()
                      for r, nt in zip(relations, node_types)}

        # the same pairs and candidates in full-KG indices for the original model
        to_full = np.vectorize(cur_to_full.get)
        idx_full = idx.clone()
        idx_full[0], idx_full[2] = torch.as_tensor(to_full(idx[0].numpy())), torch.as_tensor(to_full(idx[2].numpy()))
        rel_full = {cur["rel_token"][r]: full["rel_token"][r] for r in relations}
        idx_full[1] = torch.as_tensor([rel_full[int(z)] for z in idx[1]])
        candidates_full = {r: [cur_to_full[i] for i in c] for r, c in candidates.items()}

        for method, path, kg, x, cand in [("original KG", os.path.join(CKPT_DIR, old_name.format(seed=seed) + ".pt"), full, idx_full, candidates_full),
                                          ("curated KG", new_ckpt, cur, idx, candidates)]:
            model, h, sl = load_model(path, kg)
            for r, m in evaluate(model, h, sl, kg, x, label, flag, relations, cand).items():
                rows.append({"inference": inference, "seed": seed, "relation": r, "model": method, **m})
            del model
            torch.cuda.empty_cache()
        print(f"{inference} seed {seed} done")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "curated_vs_original_per_seed.csv"), index=False)

In [ ]:
# ── Summary: mean over seeds, paired t-test ──────────────────────────────────
summary = []
for (inference, rel), g in res.groupby(["inference", "relation"], sort=False):
    r = {"inference": inference, "relation": rel, "seeds": g["seed"].nunique()}
    for metric in ["AUROC", "AUPRC", "Recall@100"]:
        w = g.pivot_table(index="seed", columns="model", values=metric)
        r[f"{metric} original"], r[f"{metric} curated"] = w["original KG"].mean(), w["curated KG"].mean()
        r[f"{metric} p"] = stats.ttest_rel(w["curated KG"], w["original KG"]).pvalue if len(w) > 1 else np.nan
    summary.append(r)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "curated_vs_original_summary.csv"), index=False)
summary.round(4)